# 🏗️ Caterpillar Predictive Maintenance — Data Pipeline
### CWRU Bearing Dataset | Fault Detection in Heavy Machinery

**Problem:** Classify bearing faults (Normal, Inner Race, Ball, Outer Race) from high-frequency vibration signals under noisy industrial conditions, with early-stage fault detection.

**Dataset:** Case Western Reserve University (CWRU) Bearing Dataset  
**Model:** CNN-LSTM with Attention Mechanism  
**Framework:** PyTorch

## 📦 0. Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ['scipy', 'numpy', 'pandas', 'matplotlib', 'seaborn',
            'scikit-learn', 'imbalanced-learn', 'PyWavelets', 'tqdm']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All packages ready')

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import pywt

warnings.filterwarnings('ignore')
np.random.seed(42)

# Add utils to path
sys.path.append(str(Path('..').resolve()))
from utils.data_loader import load_cwru_dataset, LABEL_NAMES, LABEL_MAP
from utils.preprocessor import (
    segment_signals, zscore_normalize, add_noise,
    split_data, apply_smote, compute_fft, compute_snr
)

# ── Plot style ───────────────────────────────────────────────────────────────
BG      = '#0f1117'
FG      = '#e0e0e0'
COLORS  = ['#4fc3f7', '#66bb6a', '#ffa726', '#ab47bc']
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.edgecolor': '#444', 'axes.labelcolor': FG,
    'xtick.color': FG, 'ytick.color': FG,
    'text.color': FG, 'grid.color': '#2a2a2a',
    'font.size': 11
})

# ── Config ───────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../data/raw')
SAVE_DIR    = Path('../data/processed')
WINDOW_SIZE = 1024
OVERLAP     = 0.5
SAMPLE_RATE = 12000
RANDOM_STATE = 42
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports done')
print(f'   Data dir  : {DATA_DIR.resolve()}')
print(f'   Save dir  : {SAVE_DIR.resolve()}')
print(f'   Window    : {WINDOW_SIZE} samples @ {SAMPLE_RATE}Hz = {WINDOW_SIZE/SAMPLE_RATE*1000:.1f} ms')

---
## 📥 1. Data Loading

Download CWRU dataset from: https://engineering.case.edu/bearingdatacenter/download-data-file  
Place all `.mat` files inside `ai/data/raw/`

In [ ]:
signals, labels, metadata = load_cwru_dataset(DATA_DIR)
df_meta = pd.DataFrame(metadata)
print(f'\nMetadata shape: {df_meta.shape}')
df_meta.head(10)

---
## 🔍 2. Exploratory Data Analysis (EDA)
### 2.1 Dataset Overview

In [ ]:
print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Total recordings      : {len(signals)}')
print(f'Unique fault classes  : {len(set(labels))}')
print(f'Sampling rate         : {SAMPLE_RATE} Hz')
print(f'Window size           : {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLE_RATE*1000:.1f} ms)')
print(f'Overlap               : {int(OVERLAP*100)}%')

print('\nSignal length stats (samples):')
lengths = [len(s) for s in signals]
print(f'  Min    : {min(lengths):,}')
print(f'  Max    : {max(lengths):,}')
print(f'  Mean   : {np.mean(lengths):,.0f}')

print('\nClass distribution:')
counts = Counter(labels)
total  = sum(counts.values())
for lbl, cnt in sorted(counts.items()):
    print(f'  {LABEL_NAMES[lbl]:<15}: {cnt:>4} recordings ({cnt/total*100:.1f}%)')

### 2.2 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bearing Fault Class Distribution', fontsize=14, fontweight='bold', color=FG)

class_names = [LABEL_NAMES[k] for k in sorted(counts)]
class_counts = [counts[k] for k in sorted(counts)]

# Bar chart
ax = axes[0]
bars = ax.bar(class_names, class_counts, color=COLORS, edgecolor='#333', linewidth=0.8)
ax.set_title('Recording Count per Class', color=FG)
ax.set_ylabel('Count', color=FG)
for bar, cnt in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(cnt), ha='center', va='bottom', color=FG, fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Pie chart
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    class_counts, labels=class_names, colors=COLORS,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': BG, 'linewidth': 2}
)
for t in texts + autotexts:
    t.set_color(FG)
ax.set_title('Proportion by Class', color=FG)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Waveform Visualization

In [ ]:
# Plot one representative signal per class
class_examples = {}
for sig, lbl in zip(signals, labels):
    if lbl not in class_examples:
        class_examples[lbl] = sig
    if len(class_examples) == 4:
        break

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
fig.suptitle('Raw Vibration Waveforms — One Sample per Fault Class', fontsize=14, fontweight='bold', color=FG)

DISPLAY_SAMPLES = 3000
for ax, (lbl, sig), color in zip(axes, sorted(class_examples.items()), COLORS):
    t = np.arange(DISPLAY_SAMPLES) / SAMPLE_RATE * 1000  # ms
    ax.plot(t, sig[:DISPLAY_SAMPLES], color=color, linewidth=0.8, alpha=0.9)
    ax.set_title(f'{LABEL_NAMES[lbl]}', color=color, fontsize=11)
    ax.set_ylabel('Amplitude (g)', color=FG)
    ax.grid(alpha=0.2)
    ax.set_xlim(0, t[-1])

axes[-1].set_xlabel('Time (ms)', color=FG)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'waveforms.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 FFT Frequency Analysis

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12))
fig.suptitle('FFT Frequency Spectrum — Fault Signatures', fontsize=14, fontweight='bold', color=FG)

for ax, (lbl, sig), color in zip(axes, sorted(class_examples.items()), COLORS):
    freq, mag = compute_fft(sig[:WINDOW_SIZE], SAMPLE_RATE)
    ax.plot(freq, mag, color=color, linewidth=0.8, alpha=0.9)
    ax.set_title(f'{LABEL_NAMES[lbl]} — Frequency Domain', color=color, fontsize=11)
    ax.set_ylabel('Magnitude', color=FG)
    ax.set_xlim(0, SAMPLE_RATE / 2)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Frequency (Hz)', color=FG)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fft_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Statistical Summary

In [ ]:
stats_rows = []
for lbl, sig in sorted(class_examples.items()):
    window = sig[:WINDOW_SIZE]
    stats_rows.append({
        'Class'     : LABEL_NAMES[lbl],
        'Mean'      : f'{window.mean():.4f}',
        'Std'       : f'{window.std():.4f}',
        'Min'       : f'{window.min():.4f}',
        'Max'       : f'{window.max():.4f}',
        'RMS'       : f'{np.sqrt(np.mean(window**2)):.4f}',
        'Kurtosis'  : f'{pd.Series(window).kurtosis():.4f}',
        'Skewness'  : f'{pd.Series(window).skew():.4f}',
        'SNR (dB)'  : f'{compute_snr(window):.2f}',
        'Crest Factor': f'{np.max(np.abs(window)) / (np.sqrt(np.mean(window**2)) + 1e-8):.4f}',
    })

stats_df = pd.DataFrame(stats_rows).set_index('Class')
print('\n📊 STATISTICAL SUMMARY (per class, first window):')
print('=' * 75)
print(stats_df.to_string())

### 2.6 Fault Severity Analysis (Early Detection)

In [ ]:
fault_sizes = df_meta[df_meta['fault_type'] != 'Normal'].groupby(['fault_type', 'fault_size']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

pivot = fault_sizes.pivot(index='fault_size', columns='fault_type', values='count')
pivot.plot(kind='bar', ax=ax, color=COLORS[1:], edgecolor='#333', width=0.7)

ax.set_title('Recording Count by Fault Type and Severity', fontsize=13, fontweight='bold', color=FG)
ax.set_xlabel('Fault Size (inches)', color=FG)
ax.set_ylabel('Count', color=FG)
ax.set_xticklabels(['0.007" (Early)', '0.014" (Moderate)', '0.021" (Severe)'], rotation=0)
ax.legend(facecolor='#1e1e2e', edgecolor='#444', labelcolor=FG)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fault_severity.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n⚡ 0.007" fault size = EARLY STAGE — critical for predictive maintenance')

### 2.7 Noise Analysis (SNR per Class)

In [ ]:
snr_data = {LABEL_NAMES[lbl]: [] for lbl in set(labels)}
for sig, lbl in zip(signals, labels):
    snr_data[LABEL_NAMES[lbl]].append(compute_snr(sig[:WINDOW_SIZE]))

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

bp = ax.boxplot(
    [snr_data[LABEL_NAMES[k]] for k in sorted(set(labels))],
    labels=[LABEL_NAMES[k] for k in sorted(set(labels))],
    patch_artist=True,
    medianprops={'color': 'white', 'linewidth': 2},
)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Signal-to-Noise Ratio (SNR) per Fault Class', fontsize=13, fontweight='bold', color=FG)
ax.set_ylabel('SNR (dB)', color=FG)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'snr_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🧹 3. Data Preprocessing
### 3.1 Sliding Window Segmentation

In [ ]:
print(f'Applying sliding window: size={WINDOW_SIZE}, overlap={int(OVERLAP*100)}%')
print(f'Step size: {int(WINDOW_SIZE * (1 - OVERLAP))} samples\n')

X_raw, y = segment_signals(signals, labels, WINDOW_SIZE, OVERLAP)

# Class distribution after windowing
counts_after = Counter(y)
print('\nClass distribution after windowing:')
total_wins = len(y)
for lbl in sorted(counts_after):
    cnt = counts_after[lbl]
    print(f'  {LABEL_NAMES[lbl]:<15}: {cnt:>6,} windows ({cnt/total_wins*100:.1f}%)')

### 3.2 Z-Score Normalization

In [ ]:
X_norm = zscore_normalize(X_raw)

# Visualize effect of normalization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Before vs After Z-Score Normalization', fontsize=13, fontweight='bold', color=FG)

for idx, (ax_row, lbl) in enumerate(zip(axes, sorted(class_examples.keys())[:2])):
    sample_idx = np.where(y == lbl)[0][0]
    t = np.arange(WINDOW_SIZE) / SAMPLE_RATE * 1000
    
    ax_row[0].plot(t, X_raw[sample_idx], color=COLORS[lbl], linewidth=0.8)
    ax_row[0].set_title(f'{LABEL_NAMES[lbl]} — Raw', color=COLORS[lbl])
    ax_row[0].set_ylabel('Amplitude', color=FG)
    ax_row[0].grid(alpha=0.2)

    ax_row[1].plot(t, X_norm[sample_idx], color=COLORS[lbl], linewidth=0.8)
    ax_row[1].set_title(f'{LABEL_NAMES[lbl]} — Normalized (μ=0, σ=1)', color=COLORS[lbl])
    ax_row[1].set_ylabel('Normalized Amplitude', color=FG)
    ax_row[1].grid(alpha=0.2)

for ax in axes[-1]:
    ax.set_xlabel('Time (ms)', color=FG)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'normalization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRaw    — mean: {X_raw.mean():.4f}, std: {X_raw.std():.4f}')
print(f'Normed — mean per-sample: ~0.000, std per-sample: ~1.000')

### 3.3 Outlier Analysis

In [ ]:
# IQR-based outlier detection on window statistics
rms_vals = np.sqrt(np.mean(X_norm ** 2, axis=1))
Q1, Q3   = np.percentile(rms_vals, 25), np.percentile(rms_vals, 75)
IQR      = Q3 - Q1
lower    = Q1 - 1.5 * IQR
upper    = Q3 + 1.5 * IQR
outliers = np.sum((rms_vals < lower) | (rms_vals > upper))

print(f'RMS value stats across all windows:')
print(f'  Q1      : {Q1:.4f}')
print(f'  Q3      : {Q3:.4f}')
print(f'  IQR     : {IQR:.4f}')
print(f'  Lower   : {lower:.4f}')
print(f'  Upper   : {upper:.4f}')
print(f'  Outliers: {outliers} ({outliers/len(rms_vals)*100:.2f}%)')

# Clip outlier windows by RMS threshold
mask = (rms_vals >= lower) & (rms_vals <= upper)
X_clean = X_norm[mask]
y_clean = y[mask]
print(f'\nAfter outlier removal: {X_clean.shape[0]:,} windows (removed {(~mask).sum():,})')

### 3.4 Noise Augmentation (Simulating Industrial Conditions)

In [ ]:
# Demonstrate noise augmentation at different SNR levels
sample_sig = X_clean[np.where(y_clean == 1)[0][0]]
snr_levels = [30, 20, 10]

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle('Noise Augmentation at Different SNR Levels\n(Inner Race Fault Sample)', 
             fontsize=13, fontweight='bold', color=FG)
t = np.arange(WINDOW_SIZE) / SAMPLE_RATE * 1000

axes[0].plot(t, sample_sig, color=COLORS[1], linewidth=0.8)
axes[0].set_title('Clean Signal', color=COLORS[1])

for ax, snr, color in zip(axes[1:], snr_levels, COLORS[1:]):
    noisy = add_noise(sample_sig.reshape(1, -1), snr_db=snr)[0]
    ax.plot(t, noisy, color=color, linewidth=0.8, alpha=0.85)
    ax.set_title(f'SNR = {snr} dB', color=color)

for ax in axes:
    ax.set_ylabel('Amplitude', color=FG)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('Time (ms)', color=FG)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'noise_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training will use SNR=20dB augmentation to simulate industrial noise conditions')

---
## 🎯 4. Feature Engineering & Data Splitting
### 4.1 Train / Validation / Test Split

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X_clean, y_clean, val_size=0.15, test_size=0.15, random_state=RANDOM_STATE
)

print('\nSplit breakdown:')
for split_name, split_y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    cnt = Counter(split_y)
    print(f'\n  {split_name}:')
    for lbl in sorted(cnt):
        print(f'    {LABEL_NAMES[lbl]:<15}: {cnt[lbl]:>6,}')

### 4.2 Class Imbalance — SMOTE

In [ ]:
print('Before SMOTE:')
cnt_before = Counter(y_train)
for lbl in sorted(cnt_before):
    print(f'  {LABEL_NAMES[lbl]:<15}: {cnt_before[lbl]:,}')

X_train_res, y_train_res = apply_smote(X_train, y_train, random_state=RANDOM_STATE)

print('\nAfter SMOTE:')
cnt_after = Counter(y_train_res)
for lbl in sorted(cnt_after):
    print(f'  {LABEL_NAMES[lbl]:<15}: {cnt_after[lbl]:,}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Imbalance: Before vs After SMOTE', fontsize=13, fontweight='bold', color=FG)

for ax, cnt, title in zip(axes, [cnt_before, cnt_after], ['Before SMOTE', 'After SMOTE']):
    names = [LABEL_NAMES[k] for k in sorted(cnt)]
    vals  = [cnt[k] for k in sorted(cnt)]
    bars  = ax.bar(names, vals, color=COLORS, edgecolor='#333')
    ax.set_title(title, color=FG)
    ax.set_ylabel('Window Count', color=FG)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{v:,}', ha='center', va='bottom', color=FG, fontsize=9)
    ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Reshape for CNN-LSTM Input

In [ ]:
# CNN-LSTM expects: (batch, channels, time_steps)
# channels = 1 (single accelerometer axis)
X_train_final = X_train_res[:, np.newaxis, :]   # (N, 1, 1024)
X_val_final   = X_val[:, np.newaxis, :]           # (N, 1, 1024)
X_test_final  = X_test[:, np.newaxis, :]          # (N, 1, 1024)

print('Final tensor shapes:')
print(f'  X_train : {X_train_final.shape}')
print(f'  X_val   : {X_val_final.shape}')
print(f'  X_test  : {X_test_final.shape}')
print(f'  y_train : {y_train_res.shape}')
print(f'  y_val   : {y_val.shape}')
print(f'  y_test  : {y_test.shape}')

---
## 💾 5. Save Processed Data

In [ ]:
np.save(SAVE_DIR / 'X_train.npy', X_train_final)
np.save(SAVE_DIR / 'X_val.npy',   X_val_final)
np.save(SAVE_DIR / 'X_test.npy',  X_test_final)
np.save(SAVE_DIR / 'y_train.npy', y_train_res)
np.save(SAVE_DIR / 'y_val.npy',   y_val)
np.save(SAVE_DIR / 'y_test.npy',  y_test)

import json
meta = {
    'window_size'  : WINDOW_SIZE,
    'overlap'      : OVERLAP,
    'sample_rate'  : SAMPLE_RATE,
    'num_classes'  : 4,
    'class_names'  : [LABEL_NAMES[k] for k in sorted(LABEL_NAMES)],
    'train_samples': int(X_train_final.shape[0]),
    'val_samples'  : int(X_val_final.shape[0]),
    'test_samples' : int(X_test_final.shape[0]),
    'input_shape'  : list(X_train_final.shape[1:]),
}
with open(SAVE_DIR / 'dataset_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Saved all processed arrays to:', SAVE_DIR.resolve())
for fname in sorted(SAVE_DIR.glob('*')):
    size_mb = fname.stat().st_size / 1e6
    print(f'   {fname.name:<25} {size_mb:.2f} MB')

---
## 📝 6. Preprocessing Summary

In [ ]:
print('=' * 60)
print('  PREPROCESSING PIPELINE SUMMARY')
print('=' * 60)
print(f'  Dataset        : CWRU Bearing Dataset')
print(f'  Raw recordings : {len(signals)}')
print(f'  Window size    : {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLE_RATE*1000:.1f} ms)')
print(f'  Overlap        : {int(OVERLAP*100)}%')
print(f'  Total windows  : {len(X_clean):,}')
print(f'  After SMOTE    : {len(X_train_res):,} (train only)')
print(f'  Normalization  : Z-score per sample')
print(f'  Noise augment  : Gaussian @ SNR=20dB (during training)')
print(f'  Input shape    : {list(X_train_final.shape[1:])}  (channels × time_steps)')
print(f'  Classes        : {[LABEL_NAMES[k] for k in sorted(LABEL_NAMES)]}')
print('=' * 60)
print('\n➡️  Next: Phase 3 — CNN-LSTM Model Training')